# The PPMI data

The Parkinson's Progression Markers Initiative (PPMI) follows people with early Parkinson's disease, people at risk and healthy controls, with clinical, cognitive, imaging and biospecimen assessments at regular visits. Researchers [apply for access](https://www.ppmi-info.org/access-data-specimens/download-data), sign the data use agreement, and then download the data from the LONI Image and Data Archive (IDA).

The agreement does not allow participant-level data to be shared, so this repository holds analysis code and reviewed aggregate outputs. Download the data with your own IDA account as described below. Preserve aggregate tables and static plots when committing; keep raw data, participant identifiers and individual results out of version control. The agreement also forbids putting individual-level PPMI data into AI tools. GitHub displays the saved outputs without running the notebook, so rerun locally, review and save before publishing refreshed results.

`code/data.py` loads the downloads into an in-memory DuckDB database each time `data.PPMI()` runs, so a fresh download is picked up straight away. Run the notebooks from the `notebooks` folder after `pip install duckdb jupyterlab matplotlib numpy pandas`.

## Downloading

`dataset.ipynb` downloads what this notebook reads into the data folder: `data/` in this repository, which git ignores, or the folder named by `PPMI_DATA`.

```
data/
├── study/      Study Data CSVs, named as IDA names them
├── imaging/    image lists: search CSVs and collection lists
├── images/     unpacked DICOM and NIfTI files
├── docs/       PDFs that came with the study tables
├── ida/        IDA's list of study files (offer.csv) and the URL lists of image collections
└── downloads/  the original zips
```

This notebook needs the study tables below, the *core* set in `code/catalog.csv`. IDA adds the download date to each name, for example `Participant_Status_13Nov2026.csv`; if a table has been downloaded more than once, `data.py` uses the newest copy.

| IDA group | Tables |
|---|---|
| Subject Characteristics | Participant Status, Demographics, Socio-Economics, Age at visit |
| Motor Assessments | MDS-UPDRS Part III |
| Non-motor Assessments | Hopkins Verbal Learning Test - Revised, Benton Judgement of Line Orientation, Letter - Number Sequencing, Modified Semantic Fluency, Symbol Digit Modalities Test |
| Imaging | Xing Core Lab - Quant SBR, Grey Matter Volume, and optionally FS7 ASEG VOL and FS7 APARC CTH |

The image lists only feed the last section here and the first section of `audit.ipynb`. Without them those sections are empty and everything else still runs.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import sys
sys.path.append("../code")

import data
import plot

p = data.PPMI()
p.sql("""SELECT "table", downloaded, release, "rows", participants FROM downloads ORDER BY "table" """)

## Participants and visits

Every table names the participant in `PATNO` and the visit in `EVENT_ID`. The visits used here are SC (screening), BL (baseline), and V04, V06, V08, V10 and V12 at months 12, 24, 36, 48 and 60. Dates are recorded to the month as MM/YYYY, and the `ppmi_date` macro in `data.py` turns them into dates. This table counts the participants each table has at each of those visits.

In [ ]:
counts = " UNION ALL ".join(
    f"""SELECT '{name}' AS tbl, EVENT_ID, count(DISTINCT PATNO) AS participants FROM "{name}" GROUP BY ALL"""
    for name in p.files if "EVENT_ID" in p.db.table(name).columns)
p.sql(f"""
    SELECT tbl AS "table", coalesce(COLUMNS(* EXCLUDE (tbl)), 0) FROM (
        PIVOT ({counts}) ON EVENT_ID IN ('SC', 'BL', 'V04', 'V06', 'V08', 'V10', 'V12') USING first(participants) GROUP BY tbl
    ) ORDER BY tbl
""")

## Cohorts

`Participant_Status` assigns each person to a cohort (`COHORT` 1 is Parkinson's disease and 2 is healthy controls) and records their place in the study (`ENROLL_STATUS`). The PPMI data user guide suggests keeping participants who are enrolled, withdrew or completed. Screen failures and exclusions have no visit data.

In [ ]:
p.sql("""
    PIVOT (SELECT COHORT, COHORT_DEFINITION, lower(ENROLL_STATUS) AS status FROM participant_status)
    ON status USING count(*) GROUP BY COHORT, COHORT_DEFINITION ORDER BY COHORT, COHORT_DEFINITION
""")

## The six cognitive tests

The PPMI Cognitive Summary Score (CSS) combines six tests from five tables: HVLT-R immediate and delayed recall, Benton Judgement of Line Orientation (JLO), Letter-Number Sequencing (LNS), semantic (animal) fluency and the Symbol Digit Modalities Test (SDMT). `data.py` stacks them into one view, `cognition`, with a row per participant, visit and test. HVLT-R immediate recall is the sum of the three learning trials, so it is missing unless all three are recorded.

In [ ]:
p.sql("""
    PIVOT (SELECT test, EVENT_ID, count(DISTINCT PATNO) AS participants FROM cognition WHERE score IS NOT NULL GROUP BY ALL)
    ON EVENT_ID IN ('SC', 'BL', 'V04', 'V06', 'V08', 'V10', 'V12') USING first(participants) GROUP BY test ORDER BY test
""")

## DaTscan and grey-matter volume

`Xing_Core_Lab_-_Quant_SBR` holds DaTscan specific binding ratios for the left and right caudate and putamen, with cerebral white matter as the reference region (`*_REF_CWM`), for scans marked as analysed (`DATSCAN_ANALYZED`). `Grey_Matter_Volume` has one total grey-matter volume per MRI (`GM_VOLUME`) and the scan month (`MRIDATE`).

In [ ]:
plot.grouped(p.sql("""
    WITH scans AS (
        SELECT EVENT_ID, 'DaTscan SBR' AS measure, count(DISTINCT PATNO) AS participants
        FROM xing_core_lab_quant_sbr WHERE upper(DATSCAN_ANALYZED) = 'YES' GROUP BY ALL
        UNION ALL
        SELECT EVENT_ID, 'grey-matter volume', count(DISTINCT PATNO)
        FROM grey_matter_volume WHERE num(GM_VOLUME) > 0 GROUP BY ALL
    )
    SELECT v.EVENT_ID,
           first(participants) FILTER (measure = 'DaTscan SBR') AS "DaTscan SBR",
           first(participants) FILTER (measure = 'grey-matter volume') AS "grey-matter volume"
    FROM (VALUES ('SC'), ('BL'), ('V04'), ('V06'), ('V08'), ('V10'), ('V12')) v(EVENT_ID) LEFT JOIN scans USING (EVENT_ID)
    GROUP BY v.EVENT_ID ORDER BY visit_order(v.EVENT_ID)
"""), "Participants with a quantified scan, by visit")

## Image lists

An image search lists every image series with the participant (`Subject ID` in IDA, the same number as `PATNO`), the visit name and the modality. A collection's list, which `ida.images` saves, has the same keys but no weighting. `data.py` stacks all the lists into one table, `images`, with one row per image, and maps visit names such as *Month 12* to codes such as V04.

In [ ]:
p.sql("""
    SELECT modality, visit, event, count(*) AS series, count(DISTINCT PATNO) AS participants
    FROM images GROUP BY modality, visit, event ORDER BY modality, visit_order(event), visit
""")

## The Curated Data Cut

PPMI's Curated Data Cut is one workbook of frequently used variables, one row per participant at baseline and each annual visit. `code/curated_map.csv`, loaded as `curated_map`, says which of its columns matches which measure here:

- *same*: the same number from the same source;
- *close*: the same quantity, derived a little differently (the note says how);
- *cut only* and *ours only*: no counterpart on the other side.

The cut's data dictionary tab loads as `ppmi_curated_data_cut_public_data_dictionary`.

In [ ]:
p.sql("SELECT domain, measure, ours, cut, compare, note FROM curated_map ORDER BY entry")

`curated_check` compares the two at every visit row of the cut: rows where the cut has a value, rows where both do, and rows where they agree within the map's tolerance. Its counts are aggregates. `curated_pairs` holds the participant-level values behind them, so it stays on your machine like any other PPMI table. Both are empty until the cut is in `data/study/`.

In [ ]:
p.sql("""
    SELECT domain, measure, cut, compare, in_cut, in_both, agree,
           round(100.0 * agree / nullif(in_both, 0), 1) AS "agree %",
           round(median_difference, 3) AS "median difference", round(largest_difference, 3) AS "largest difference"
    FROM curated_check
""")

`COG_COMPOSITE_INT` is PPMI's own Cognitive Summary Score: the same six tests and the same paper (Weintraub et al. 2025), with norms from PPMI's healthy controls. Score the CSS here with the paper's published norms and compare. Close agreement says the scoring is right; a steady offset points to the norms or to the cut's cap of 20 years on education.

In [ ]:
from ppmi_pipeline import css

css_check = "Put the Curated Data Cut in data/study/ to compare the two scores."
if "ppmi_curated_data_cut_public" in p.files:
    p.db.execute("USE clean")
    try:
        css.add_css_view(p.db, css.PROJECT / "css_reference_norms.csv")
        css_check = p.sql("""
            SELECT count(*) AS visits, count(*) FILTER (abs(s.css - c.composite) <= 0.01) AS "within 0.01",
                   round(median(abs(s.css - c.composite)), 3) AS "median difference",
                   round(corr(s.css, c.composite), 3) AS correlation
            FROM css_scores s
            JOIN (SELECT PATNO, EVENT_ID, TRY_CAST(COG_COMPOSITE_INT AS DOUBLE) AS composite
                  FROM memory.ppmi_curated_data_cut_public) c ON c.PATNO = s.patient_id AND c.EVENT_ID = s.event_id
            WHERE s.css IS NOT NULL AND c.composite IS NOT NULL
        """)
    finally:
        p.db.execute("USE memory")
css_check